# RockID · Phase 2 · Evaluation and go/no-go gate

Scores the model trained by `02_train` and decides whether it is good enough for the web app.

1. **Calibration** on validation: a temperature that makes confidences honest, and the thresholds for the app's "uncertain" result (top-1 below τ, or top-1 and top-2 too close).
2. **Test set**, preprocessed exactly like the app (center square crop, bilinear resize), scored once.
3. **Field set**: your own photos in `MyDrive/RockID/field_test/<class>/`, minus near-copies of dataset images.
4. **Gate** (docs/PHASES.md): test top-1 ≥ 75 %, test top-3 ≥ 90 %, field top-3 ≥ 70 %, no class recall < 50 %.

Nothing here is fitted on test or field data. Runs on CPU too, but a T4 is faster.

**Outputs** go to `MyDrive/RockID/runs/<run_name>/eval/`; step 7 downloads the parts that belong in the repo.

## 1 · Settings

In [ ]:
REPO_URL = "https://github.com/GrowingRui/RockID.git"   # empty = use MyDrive/RockID/code/ml instead
REPO_BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/RockID"
CONFIG_FILE = "configs/train_v1.json"   # the training config of the run to evaluate
TARGET_CONFIDENT_ACC = 0.90   # accuracy the app should have when it does NOT say "uncertain"
N_GOLDEN = 50                 # test images kept as the parity set for export (Phase 3) and browser (Phase 6)

## 2 · Setup: Drive, code, trained model

In [ ]:
import importlib, json, os, pathlib, shutil, subprocess, sys, zipfile
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = pathlib.Path(DRIVE_ROOT)
CACHE = DRIVE / "cache"

if REPO_URL:
    CODE = pathlib.Path("/content/rockid")
    url = REPO_URL
    try:
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass  # public repo, no token needed
    cmd = ["git", "-C", str(CODE), "pull", "--ff-only"] if CODE.exists() else \
          ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, str(CODE)]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        raise RuntimeError("git clone/pull failed. Check REPO_URL, REPO_BRANCH and the GITHUB_TOKEN secret.") from None
    ML = CODE / "ml"
else:
    ML = DRIVE / "code" / "ml"
sys.path.insert(0, str(ML))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagehash>=4.3"], check=True)

import numpy as np, pandas as pd
import tensorflow as tf
import rockid_ml
from rockid_ml import classes, dedup, images, metrics, prep, probes, report, scan, train, evaluate
for m in (images, scan, dedup, report, probes, classes, metrics, prep, train, evaluate):
    importlib.reload(m)

cfg = train.load_config(ML / CONFIG_FILE)
RUN = DRIVE / "runs" / cfg.run_name
assert (RUN / "model.keras").exists(), f"no trained model in {RUN}; run 02_train first"
saved_cfg = json.loads((RUN / "config.json").read_text())
assert saved_cfg == json.loads(json.dumps(cfg.to_dict())), "config file differs from the one this run was trained with"

model = tf.keras.models.load_model(RUN / "model.keras")
LABELS = json.loads((RUN / "labels.json").read_text())
OUT = RUN / "eval"; FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
print("run", cfg.run_name, "·", len(LABELS), "classes · TF", tf.__version__,
      "· GPU:", tf.config.list_physical_devices("GPU") or "none (fine, just slower)")

## 3 · Calibration and "uncertain" thresholds (validation only)

In [ ]:
manifest = pd.read_csv(ML / cfg.split_manifest)
assert sorted(manifest["label"].unique()) == LABELS, "labels.json does not match the split manifest"
LABEL_ID = {l: i for i, l in enumerate(LABELS)}
va = manifest[manifest["split"] == "val"].reset_index(drop=True)

v = np.load(RUN / "val_logits.npz", allow_pickle=True)
assert list(v["rel_path"]) == va["rel_path"].tolist(), "val_logits.npz is out of sync with the manifest"
val_logits, y_val = v["logits"], v["y"]

T = evaluate.fit_temperature(val_logits, y_val)
p_val_raw, p_val = evaluate.softmax(val_logits), evaluate.softmax(val_logits, T)
thr = evaluate.fit_thresholds(p_val, y_val, TARGET_CONFIDENT_ACC)
calib = {
    "temperature": T, "tau": thr["tau"], "margin": thr["margin"], "target_acc": TARGET_CONFIDENT_ACC,
    "target_met": thr["target_met"], "val_coverage": thr["coverage"], "val_confident_acc": thr["confident_acc"],
    "val_ece_before": evaluate.ece(p_val_raw, y_val), "val_ece_after": evaluate.ece(p_val, y_val),
    "val_nll_before": evaluate.nll(val_logits, y_val), "val_nll_after": evaluate.nll(val_logits, y_val, T),
}
print(f"T = {T:.3f} · ECE {calib['val_ece_before']:.3f} → {calib['val_ece_after']:.3f}")
print(f"uncertain if top-1 < {thr['tau']:.2f} or margin < {thr['margin']:.2f}: "
      f"{thr['coverage']:.0%} confident at {thr['confident_acc']:.1%} (target {TARGET_CONFIDENT_ACC:.0%}, "
      f"{'met' if thr['target_met'] else 'NOT reachable'})")

## 4 · Test set (preprocessed like the app)

In [ ]:
WORK = pathlib.Path("/content/data"); WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / "raw"
if not DATA.exists():
    ZIP_CACHE = CACHE / "dataset.zip"
    if not ZIP_CACHE.exists():
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME").strip()
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY").strip()
        subprocess.run(["kaggle", "datasets", "download", "-d", "kutlayusta/type-of-rocks-and-minerals-dataset",
                        "-p", str(WORK), "--force"], check=True)
        shutil.copy(next(WORK.glob("*.zip")), ZIP_CACHE)
    with zipfile.ZipFile(ZIP_CACHE) as z:
        z.extractall(DATA)

def load_images(paths):
    with ThreadPoolExecutor(8) as ex:
        return np.stack(list(ex.map(lambda p: prep.client_preprocess(p, cfg.image_size), paths)))

def predict(paths):
    return model.predict(load_images(paths).astype("float32"), batch_size=64, verbose=0)

te = manifest[manifest["split"] == "test"].reset_index(drop=True)
y_test = te["label"].map(LABEL_ID).to_numpy()
test_logits = predict([str(DATA / p) for p in te["rel_path"]])
p_test = evaluate.softmax(test_logits, T)
test = evaluate.score(y_test, p_test, LABELS)
sel = evaluate.selective(p_test, y_test, thr["tau"], thr["margin"])
calib.update(test_coverage=sel["coverage"], test_confident_acc=sel["confident_acc"], test_uncertain_acc=sel["uncertain_acc"])
np.savez_compressed(OUT / "test_logits.npz", logits=test_logits, y=y_test, rel_path=te["rel_path"].to_numpy())
print(f"test top-1 {test['top1']:.1%} · top-3 {test['top3']:.1%} · lowest recall {test['min_recall']:.0%} ({test['min_recall_class']})")
print(f"confident on {sel['coverage']:.0%} of test photos, {sel['confident_acc']:.1%} of those correct")

## 5 · Field set

Folder names are mapped through the class list's merges (a `flint/` folder counts as `chert`). Photos of classes the model doesn't know are listed but not scored, and near-copies of dataset images (pHash, same threshold as the audit) are left out.

In [ ]:
FIELD = DRIVE / "field_test"
field = None
if FIELD.exists() and any(FIELD.iterdir()):
    cc = classes.load_class_config(ML / cfg.class_file)
    f = scan.scan_dataset(FIELD)
    f["label_final"] = f["label"].map(lambda l: cc["merges"].get(l, l))
    f["known_class"] = f["label_final"].isin(LABEL_ID)
    f["phash"] = [images.analyze_image(FIELD / p).get("phash") for p in f["rel_path"]]
    f = f[f["phash"].notna()].reset_index(drop=True)
    stats_csv = CACHE / "image_stats.csv"
    if stats_csv.exists():
        ds_phash = pd.read_csv(stats_csv, usecols=["phash"])["phash"].dropna()
        f["leaks_into_dataset"] = evaluate.field_leaks(f["phash"], ds_phash)
    else:
        print("WARNING: audit cache image_stats.csv not found, field leak check skipped")
        f["leaks_into_dataset"] = False
    use = f[f["known_class"] & ~f["leaks_into_dataset"]].reset_index(drop=True)
    if len(use):
        y_field = use["label_final"].map(LABEL_ID).to_numpy()
        p_field = evaluate.softmax(predict([str(FIELD / p) for p in use["rel_path"]]), T)
        field = {**evaluate.score(y_field, p_field, LABELS), "n_leaks": int(f["leaks_into_dataset"].sum()),
                 "n_unknown": int((~f["known_class"]).sum())}
        use["pred"] = np.asarray(LABELS)[p_field.argmax(1)]
        use["p1"] = p_field.max(1).round(3)
        use["uncertain"] = evaluate.uncertain_mask(p_field, thr["tau"], thr["margin"])
        use[["rel_path", "label_final", "pred", "p1", "uncertain"]].to_csv(OUT / "field_predictions.csv", index=False)
        print(f"field: {len(use)} photos · top-1 {field['top1']:.1%} · top-3 {field['top3']:.1%}")
    print(f"{len(f)} field photos · {int(f['leaks_into_dataset'].sum())} near-copies of dataset images · "
          f"unknown classes: {sorted(f.loc[~f['known_class'], 'label'].unique()) or 'none'}")
else:
    print("No field set at", FIELD, "· the gate cannot pass without it")

## 6 · Gate

In [ ]:
gate_df = evaluate.gate(test, field)
cm = evaluate.confusion(y_test, test["pred"], len(LABELS))
advice = evaluate.weak_class_advice(test["per_class_recall"], cm, LABELS, metrics.GATE["min_recall"])
passed = (gate_df["result"] == "pass").all()
print("GATE PASSED" if passed else "GATE NOT PASSED")
for a in advice:
    print("•", a)
gate_df

## 7 · Report, parity set and download

In [ ]:
figures = {
    "test confusion matrix": FIG / "confusion.png",
    "test recall per class": FIG / "test_recall.png",
    "calibration": FIG / "reliability.png",
    "uncertain threshold": FIG / "coverage.png",
}
evaluate.plot_confusion(cm, LABELS, figures["test confusion matrix"])
table = pd.DataFrame({"recall": test["per_class_recall"],
                      "test_images": te["label"].value_counts().reindex(LABELS),
                      "val_recall": metrics.per_class_recall(y_val, p_val.argmax(1), LABELS)}, index=LABELS)
if field:
    table["field_recall"] = field["per_class_recall"]
table = table.sort_values("recall")
report.plot_share(table, "recall", figures["test recall per class"], "test recall", metrics.GATE["min_recall"])
evaluate.plot_reliability(evaluate.reliability(p_val_raw, y_val), evaluate.reliability(p_val, y_val), figures["calibration"], T)
evaluate.plot_coverage(p_val, y_val, {**thr, "target_acc": TARGET_CONFIDENT_ACC}, figures["uncertain threshold"])
confused = probes.confused_pairs(te["label"].to_numpy(), np.asarray(LABELS)[test["pred"]])

info = {
    "Model": f"{cfg.arch} α {cfg.alpha}, {cfg.image_size} px, {len(LABELS)} classes (run `{cfg.run_name}`)",
    "Validation (training view)": f"top-1 {metrics.topk_accuracy(y_val, p_val, 1):.1%} · top-3 {metrics.topk_accuracy(y_val, p_val, 3):.1%}",
    "Test": f"top-1 {test['top1']:.1%} · top-3 {test['top3']:.1%} on {test['n']:,} images",
    "Field": "missing" if not field else f"top-1 {field['top1']:.1%} · top-3 {field['top3']:.1%} on {field['n']} photos",
}
path = evaluate.write_report(OUT, cfg.run_name, info, gate_df, calib, test, field, table, confused, advice, figures)
table.to_csv(OUT / "test_per_class.csv")
(OUT / "calibration.json").write_text(evaluate.calibration_json(calib, LABELS))
(OUT / "gate.json").write_text(json.dumps({"passed": bool(passed), "checks": gate_df.to_dict("records")}, indent=2, default=float))
golden = evaluate.golden_set(te, p_test, LABELS, n=N_GOLDEN, seed=cfg.seed)
golden.to_csv(OUT / "golden_set.csv", index=False)
print("Report:", path)

from google.colab import files
bundle = WORK / f"rockid_eval_{cfg.run_name}.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    base_dir = f"reports/eval-{cfg.run_name}"
    for name in ("eval.md", "test_per_class.csv", "calibration.json", "gate.json", "golden_set.csv"):
        z.write(OUT / name, f"{base_dir}/{name}")
    for f_ in FIG.glob("*.png"):
        z.write(f_, f"{base_dir}/figures/{f_.name}")
files.download(str(bundle))
print("Unzip into the repo's ml/ folder. field_predictions.csv stays on Drive (your photos' file names).")

## Next
- **Gate passed:** commit `reports/eval-<run_name>/` and go to Phase 3 (`04_export`), which bakes the temperature into the model and writes `calibration.json` into `metadata.json`.
- **Gate not passed:** see *Weak classes* in the report. The options from ADR 0001 are more data (Wikimedia Commons, your own photos), merging or dropping classes (new `classes_v2.json`, re-run the audit split), or a lower gate with a stronger "uncertain" result. Record the decision in an ADR.